# Fuentes de Datos para Ciencia de Datos e Ingeniería

## 🎯 Objetivos de Aprendizaje
- Conocer las principales fuentes de datos públicas y repositorios abiertos.
- Aprender a consumir datos vía APIs REST con autenticación y buenas prácticas.
- Integrar flujos de ingesta de datos tabulares (CSV, Excel, Parquet, JSON) usando Pandas y Pathlib.
- Diseñar un pipeline automatizado para descargar y validar datos de fuentes remotas.
- Aplicar Python 3.12+ para manejo robusto de URLs y archivos en disco.

## 🌉 Puente Pedagógico: La Materia Prima del Análisis

### ¿Por qué importan las fuentes de datos?
Ningún modelo predictivo o análisis estadístico tiene valor sin datos confiables. Aprender a buscar, autenticarse y extraer datos de repositorios confiables es el primer eslabón del ciclo de vida de datos.

### Analogía
Los datos son como el agua potable:
- **Fuentes naturales**: APIs abiertas y repositorios abiertos (Kaggle, Banco Mundial, Gobiernos abiertos).
- **Tratamiento y purificación**: Descarga, parsing y validación de esquema con Pandas.
- **Distribución**: Almacenamiento local estructurado o base de datos analítica.

### Diagrama de Ingesta y Consumo
```
  +--------------------+        +-------------------+
  |   Kaggle / APIs    |        | Archivos Abiertos |
  | (World Bank, INEGI)|        |   (CSV / JSON)    |
  +---------+----------+        +---------+---------+
            |                             |
            +--------------+--------------+
                           |
                           v
           [ Ingesta Python (requests / urllib) ]
                           |
                           v
            [ Validación de Esquema (Pandas) ]
                           |
                           v
           [ Almacenamiento Local (Parquet / SQLite) ]
```

In [ ]:
from pathlib import Path
import pandas as pd
import json
import urllib.request

# Creación de directorio de descargas local estructurado
DATA_DIR = Path("./data_cache")
DATA_DIR.mkdir(exist_ok=True)
print(f"Directorio local de datos listo: {DATA_DIR.resolve()}")

## 1. Consumo de APIs Abiertas (Ejemplo: Banco Mundial)

El Banco Mundial provee una API pública sin necesidad de registro previo que entrega indicadores socioeconómicos mundiales en formato JSON.

In [ ]:
# API del Banco Mundial: PIB per cápita (NY.GDP.PCAP.CD) para países de América del Norte
api_url = "http://api.worldbank.org/v2/country/USA;MEX;CAN/indicator/NY.GDP.PCAP.CD?date=2018:2022&format=json"

try:
    req = urllib.request.Request(api_url, headers={"User-Agent": "Python-Data-Course"})
    with urllib.request.urlopen(req, timeout=10) as response:
        raw_data = json.loads(response.read().decode("utf-8"))
        
    # La API del Banco Mundial devuelve [metadata, records]
    registros = raw_data[1]
    df_gdp = pd.json_normalize(registros)
    df_limpio = df_gdp[["country.value", "date", "value"]].rename(
        columns={"country.value": "pais", "date": "anio", "value": "pib_per_capita_usd"}
    )
    print("--- Datos del Banco Mundial Cargados ---")
    display(df_limpio.head(6))
except Exception as e:
    print(f"Nota: Ejecución offline o timeout: {e}")

## 2. Fuentes de Datos Populares y Formatos de Almacenamiento

| Repositorio / Fuente | Tipo de Acceso | Especialidad | Formatos Habituales |
|:---|:---|:---|:---|
| **Kaggle Datasets** | API / Web / CLI | Competencias, ML, Tabular | CSV, JSON, Parquet, SQLite |
| **World Bank Open Data** | REST API abierta | Indicadores económicos globales | JSON, XML, CSV |
| **UCI Machine Learning Rep.** | HTTP Directo | Benchmarks clásicos de ML | CSV, Arff, TXT |
| **Data.gov / Datos Abiertos** | Portales de Gobierno | Censos, transporte, salud | CSV, GeoJSON, SHP |
| **Hugging Face Datasets** | Librería Python (`datasets`) | NLP, Visión, Audio, LLMs | Parquet, Arrow |

## 📝 Ejercicios Prácticos

### Ejercicio 1 (Guiado): Carga y normalización de JSON local simulado

In [ ]:
# Simular respuesta de una API de e-commerce
datos_api = [
    {"id_pedido": 101, "cliente": "Ana", "items": [{"prod": "Laptop", "precio": 1200}, {"prod": "Mouse", "precio": 25}]},
    {"id_pedido": 102, "cliente": "Carlos", "items": [{"prod": "Teclado", "precio": 75}]}
]

# Aplanar registros anidados con pd.json_normalize
df_pedidos = pd.json_normalize(datos_api, record_path=["items"], meta=["id_pedido", "cliente"])
display(df_pedidos)

### Ejercicio 2 (Independiente): Exportación a Parquet para Eficiencia
Exporta `df_pedidos` a formato Parquet en `data_cache/pedidos.parquet` y compara el tipo de dato recuperado.

In [ ]:
target_file = DATA_DIR / "pedidos.parquet"
try:
    df_pedidos.to_parquet(target_file, index=False)
    df_recuperado = pd.read_parquet(target_file)
    print("Archivo Parquet generado y re-leído exitosamente:")
    display(df_recuperado)
except ImportError:
    print("pyarrow o fastparquet no instalado en este ambiente. Guardando en CSV alternativo:")
    df_pedidos.to_csv(DATA_DIR / "pedidos.csv", index=False)
    print("Guardado en CSV exitosamente.")

## 📋 Resumen
- Las APIs REST con salida JSON son el mecanismo estándar para alimentar pipelines de datos dinámicos.
- `pd.json_normalize()` simplifica el procesamiento de datos semiestructurados con jerarquías anidadas.
- Formatos binarios modernos como Parquet ofrecen compresión superior y retención exacta de tipos de datos respecto a CSV.